In [11]:
import sys
sys.path.append('/home/chinahg/GCresearch/contrailuncertainty/start_here/')
import pipeline_fxn_lib as lib
sys.path.append('/home/chinahg/GCresearch/contrailuncertainty/LRT/')
import LRT_fxnlib as LRTlib

import os
import yaml
import shutil
import time

import numpy as np
import xarray as xr
import pandas as pd

In [12]:
test_ids = ['130T205L25'] #['110T205L25', '110T218L25', '110T225L25', '130T205L25', '130T218L25', '130T225L25']
timestep_seconds = [45, 30, 20, 40, 24, 16]
num_sims = len(test_ids)
TRANSPORT_TIMESTEP = 60/60 # Minutes, default is 1 minute
ICE_GROWTH_TIMESTEP = TRANSPORT_TIMESTEP # Minutes

COAG = 'F'  # Whether to enable coagulation
COAG_TIMESTEP = TRANSPORT_TIMESTEP # Minutes

TEMP_PERTURB = True  # Whether to enable temperature perturbation
TEMP_AMP = 6   # Amplitude of temperature perturbation [K], default is 0.2 K
TEMP_TIMESTEP = 10     # Timescale for temperature perturbation in minutes, default is the same as transport timestep. Must be the same or larger than transport timestep.
SEED_VALUE = 3      # Seed value for random number generation
DH = 15 # Horizontal diffusion coefficient in m^2/s, default is 15 m^2/s
DV = 0.15 # Vertical diffusion coefficient in m^2/s, default is 0.15 m^2/s
NX = 200  # Max number of grid points in the horizontal direction, default is 200
NY = 180  # Max number of grid points in the vertical direction, default is 180
NUM_THREADS = 8  # Number of threads to use for parallel processing, default is 4 (choose 1 to remove any parallelization)

# Where is the base YAML file located?
source_path = "/home/chinahg/GCresearch/contrailuncertainty/APCEMM_vs_CoCiP_vs_LES/APCEMM/base_inputs/B767_LES_CoCiP_APCEMM_input.yaml"  # Source YAML file

# Where do you want to save the modified YAML files and simulation outputs?
binning = True  # Set to True for binning test, False for matching test (we need to know if we need to rebin the epm-input.nc file)
test_identifier = f"1000-bins-TP6-TT1-threaded"  # f'nx{NX}_ny{NY}'  # If binning is True, should be of the form (number of bins)-bins         # f'vdiff_{DV}_hdiff_{DH}' #f'{TEMP_SCALE}min_TP_{TEMP_AMP}K_s{SEED_VALUE}'
save_directory = f"/home/chinahg/GCresearch/contrailuncertainty/APCEMM_vs_CoCiP_vs_LES/APCEMM/testing/{test_identifier}"

# Where are the base netCDF meteorological files and bypass files located?
base_file_dir = "/home/chinahg/GCresearch/contrailuncertainty/APCEMM_vs_CoCiP_vs_LES/APCEMM/base_inputs/"

# Make the epm-bypass.nc file

In [13]:
def create_epm_input_netcdf(test_id):    
    path = f"/home/chinahg/GCresearch/contrailuncertainty/APCEMM_vs_CoCiP_vs_LES/APCEMM/base_inputs/{test_id}/epm-input.nc"
    with xr.open_dataset(path) as ds:
        original_epm_bypass_dataset = ds.load()

    # Import LLES data to match
    LES_data = pd.read_csv(f"/home/chinahg/GCresearch/contrailuncertainty/APCEMM_vs_CoCiP_vs_LES/LES/processed_data/all_data/{test_id}.csv")
    LES_initial_mass_per_m = LES_data['Ice_mass'].values[0]
    LES_initial_number_per_m = LES_data['Ice_number'].values[0]
    LES_initial_area = LES_data['Area_m2'].values[0]
    LES_initial_effective_radius = LES_data['Effective_radius_um'].values[0]
    target_sauter_means = {
        "110T205L25": 1.2,
        "110T218L25": 3.4,
        "110T225L25": 5.8,
        "130T205L25": 1.8,
        "130T218L25": 4.4,
        "130T225L25": 5.8
    }
    target_sauter_mean = target_sauter_means.get(test_id, 0.0)

    if test_id.startswith("110"):
        LES_initial_PSD = pd.read_csv("/home/chinahg/GCresearch/contrailuncertainty/APCEMM_vs_CoCiP_vs_LES/LES/raw_data_from_plots/ice_crystal_radius/110T205L25/110T205L25_5min.csv")
    else:
        LES_initial_PSD = pd.read_csv("/home/chinahg/GCresearch/contrailuncertainty/APCEMM_vs_CoCiP_vs_LES/LES/raw_data_from_plots/ice_crystal_radius/130T225L25/130T225L25_5min.csv")

    # Horizontal shift only (no PSD amplitude scaling): scale radius axis
    original_sauter_mean = LRTlib.calculate_sauter_mean(
        LES_initial_PSD.iloc[:, 0].values,
        LES_initial_PSD.iloc[:, 1].values
    )
    radius_scale_factor = target_sauter_mean / original_sauter_mean
    LES_initial_PSD.iloc[:, 0] = LES_initial_PSD.iloc[:, 0].values * radius_scale_factor

    current_sauter_mean = LRTlib.calculate_sauter_mean(
        LES_initial_PSD.iloc[:, 0].values,
        LES_initial_PSD.iloc[:, 1].values
    )

    # print(f"Initial LES quantities for {test_id}:\n")
    # print(f"Initial Ice Mass per m: {LES_initial_mass_per_m:.4f} kg/m")
    # print(f"Initial Ice Number per m: {LES_initial_number_per_m:.4f} #/m")
    # print(f"Initial Area: {LES_initial_area:.4f} m²")
    # print(f"Initial Effective Radius: {LES_initial_effective_radius:.4f} μm")
    # print(f"Shifted Initial Sauter Mean Radius: {current_sauter_mean:.2f} μm")

    if binning == True:
        print("Running binning test...")
        num_bins = int(test_identifier.split('-')[0])  # Set the desired number of bins for the test (only active if binning = "True")

        # For binning test: update the bin edges to match the 15-bin configuration
        # The bin edges start at 50nm and the bin centers increase by a factor of 2^1/3 (doubling mass) each time
        original_bin_edges = original_epm_bypass_dataset['ice_r_e'].values
        original_bin_centers = original_epm_bypass_dataset['ice_r'].values
        original_num_bins = len(original_bin_centers)

        new_bin_centers = np.zeros(num_bins)
        new_bin_widths = np.zeros(num_bins)
        new_bin_edges = np.zeros(num_bins + 1)

        new_bin_edges[0] = original_bin_edges[0]  # Start at 50nm
        new_bin_edges[-1] = original_bin_edges[-1]  # End at 1mm
        new_bin_centers[0] = original_bin_centers[0]  # Start at 50nm

        K0 = 1.21644  # the original factor by which the radius increases for each subsequent bin center
        K = K0**((original_num_bins - 1)/(num_bins -1))  # Calculate the radius ratio to maintain the same range with fewer bins
        print(f"Original number of bins: {original_num_bins}, New number of bins: {num_bins}, Original K: {K0:.5f}, New K: {K:.5f}")

        for i in range(1, num_bins):
            new_bin_centers[i] = new_bin_centers[i-1] * K
            new_bin_edges[i] = new_bin_edges[0] * K**(i)  # Calculate the new bin edges based on the new bin centers
        
        print(f"New bin edges: {new_bin_edges}")
        print(f"New bin centers: {new_bin_centers}")
        
        # Make a copy of the EPM dataset to edit and save at the end of the script
        new_epm_bypass_dataset = original_epm_bypass_dataset.copy()

        # for binning test: update the bin edges to match the 15-bin configuration
        # Delete the existing ice_r_e and ice_r variables and replace with the new bin edges and centers
        new_epm_bypass_dataset = new_epm_bypass_dataset.drop_vars(['ice_r_e', 'ice_r', 'ice_pdf'])

        # Assign new coordinates/dimensions directly — does NOT affect other variables
        new_epm_bypass_dataset = new_epm_bypass_dataset.assign_coords({
            'ice_r': new_bin_centers,
            'ice_r_e': new_bin_edges,
        })

        # Create new variable for ice_pdf with the correct dimensions and values
        new_epm_bypass_dataset['ice_pdf'] = (('ice_r'), np.zeros(num_bins))  # Create a new ice_pdf variable with the correct number of bins
        # Create new r_ice_e and r_ice coordinates with the new bin edges and centers
        new_epm_bypass_dataset['ice_r_e'] = (('ice_r_e'), new_bin_edges)
        new_epm_bypass_dataset['ice_r'] = (('ice_r'), new_bin_centers)

    else:
        # Make a copy of the EPM dataset to edit and save at the end of the script
        new_epm_bypass_dataset = original_epm_bypass_dataset.copy()

    LLES_sauter_mean = LRTlib.calculate_sauter_mean(LES_initial_PSD.iloc[:,0].values, LES_initial_PSD.iloc[:,1].values)
    new_epm_bypass_dataset['iceRadius'] = LLES_sauter_mean * 1e-6  # Convert to meters

    ### Convert the LES PSD datapoints onto the EPM bin system
    # Bin the LLES data using the specified bin edges
    bin_edges = new_epm_bypass_dataset['ice_r_e'].values * 1e6  # Convert to µm
    bin_indices = np.digitize(LES_initial_PSD.iloc[:,0], bin_edges) - 1

    # Prepare arrays to store mean radius and mean value per bin
    mean_radii = []
    mean_values = []
    bin_width = []

    for i in range(len(bin_edges) - 1):
        mask = bin_indices == i
        if np.any(mask):
            bin_width.append(np.log(bin_edges[i+1]) - np.log(bin_edges[i]))
            mean_radii.append(new_epm_bypass_dataset['ice_r'].values[i] * 1e6)  # Mean radius is the center of the bin
            mean_values.append(LES_initial_PSD.iloc[:,1][mask].mean())
        else:
            bin_width.append(np.log(bin_edges[i+1]) - np.log(bin_edges[i]))
            mean_radii.append(new_epm_bypass_dataset['ice_r'].values[i] * 1e6)  # Mean radius is the center of the bin
            mean_values.append(0.0)

    ### Scale the LES PSD to match the total number of ice particles in the LES dataset
    ice_pdf = np.array(mean_values)      # [#/cm³] per ln(r)
    ice_r_e = new_epm_bypass_dataset['ice_r_e'].values  # [m]
    log_bin_widths = np.log(ice_r_e[1:] / ice_r_e[:-1])
    area = new_epm_bypass_dataset['area'].values

    # Calculate total particles
    moment_0 = np.sum(ice_pdf * log_bin_widths)
    total_particles = moment_0 * area * 1e6 # #/m
    total_req_particles = LES_initial_number_per_m
    num_scaling = total_req_particles / (moment_0 * area * 1e6)
    num_engines = 2
    new_epm_bypass_dataset['ice_pdf'].values = num_scaling * ice_pdf * (1/num_engines)

    ice_pdf = new_epm_bypass_dataset['ice_pdf'].values      # [#/cm³] per ln(r)
    # print(f"Total particles after scaling: {np.sum(ice_pdf * np.log(ice_r_e[1:] / ice_r_e[:-1])) * area * 1e6:.3e} [#/m]")

    ### Calculate total ice mass from the scaled PSD
    ### Ice mass calculation
    ice_r = new_epm_bypass_dataset['ice_r'].values # [m]

    # 3rd moment
    moment_3 = np.sum(ice_pdf * (ice_r**3) * log_bin_widths)

    # Convert to volume concentration [m³/cm³]
    volume_factor = (4.0/3.0) * np.pi
    volume_concentration = volume_factor * moment_3  # [m³/cm³]
    total_volume_per_m = volume_concentration * area * 1e6  # [m³/m]

    RHO_ICE = 917  # [kg/m³]
    total_mass = total_volume_per_m * RHO_ICE  # [kg/m]
    two_engine_total_mass = 2 * total_mass

    saved_total_ice_num = 2 * np.sum(new_epm_bypass_dataset['ice_pdf'].values * np.log(new_epm_bypass_dataset['ice_r_e'].values[1:] / new_epm_bypass_dataset['ice_r_e'].values[:-1]) * new_epm_bypass_dataset['area'].values * 1e6)

    # print(f"New APCEMM EPM bypass total ice number, both engines [# m^-1]: {saved_total_ice_num:.3e}")
    # print(f"LES initial total ice number, both engines [# m^-1]: {LES_initial_number_per_m:.3e}")

    # print(f"New APCEMM EPM bypass initial ice mass per m: {two_engine_total_mass:.4f} kg/m")
    # print(f"LES initial ice mass per m: {LES_initial_mass_per_m:.4f} kg/m")

    # print(f"New APCEMM EPM bypass initial effective radius: {new_epm_bypass_dataset['iceRadius'].values *1e6:.2f} µm")
    # print(f"LES initial effective radius: {current_sauter_mean:.2f} µm")

    # # Save the changes to the original_epm_bypass_dataset
    new_epm_bypass_dataset.to_netcdf((f"/home/chinahg/GCresearch/contrailuncertainty/APCEMM_vs_CoCiP_vs_LES/APCEMM/base_inputs/{test_id}/epm-input-{test_identifier}.nc"))

    return None


# Make YAML file

In [14]:
# Make YAML files point to the correct .nc file
# YAML_paths = []
# overlay_paths = []

def create_yaml(i):
    test_id = test_ids[i]
    print(f"Processing test ID: {test_id}")

    ## For input YAML
    base_destination_dir = f"{save_directory}/{test_id}"  # Target directory
    YAML_destination_path = os.path.join(base_destination_dir, "B767_LES_CoCiP_APCEMM_input.yaml")
    print(f"Destination path: {YAML_destination_path}")

    # Create the destination directory if it doesn't exist
    if not os.path.exists(base_destination_dir):
        print(f"Creating directory: {base_destination_dir}")
        os.makedirs(base_destination_dir, exist_ok=False)

    # Copy the YAML file to the new directory
    shutil.copy(source_path, YAML_destination_path)

    # Read and modify the copied YAML file
    with open(YAML_destination_path, "r") as file:
        data = yaml.safe_load(file)  # Load YAML into a Python dictionary
    
    ## Make an output directory to store results
    output_dir = os.path.join(base_destination_dir, "outputs")
    if not os.path.exists(output_dir):
        print(f"Creating output directory: {output_dir}")
        os.makedirs(output_dir, exist_ok=False)

    # Modify the YAML content
    data["SIMULATION MENU"]["OUTPUT SUBMENU"]["Output folder (string)"] = output_dir
    data["METEOROLOGY MENU"]["METEOROLOGICAL INPUT SUBMENU"]["Met input file path (string)"] = f"{base_file_dir}/{test_id}/{test_id}.nc"
    data["TRANSPORT MENU"]["Transport Timestep [min] (double)"] = TRANSPORT_TIMESTEP
    data["AEROSOL MENU"]["Ice growth timestep [min] (double)"] = ICE_GROWTH_TIMESTEP
    data["AEROSOL MENU"]["Coag. timestep [min] (double)"] = COAG_TIMESTEP
    data["AEROSOL MENU"]["Turn on solid coagulation (T/F)"] = COAG
    data["AEROSOL MENU"]["Turn on liquid coagulation (T/F)"] = COAG
    data["PARAMETER MENU"]["METEOROLOGICAL PARAMETERS SUBMENU"]["Horiz. diff. coeff. [m^2/s] (double)"] = DH
    data["PARAMETER MENU"]["METEOROLOGICAL PARAMETERS SUBMENU"]["Vert. diff. coeff. [m^2/s] (double)"] = DV # Two inputs??? The first seems to not be used...
    data["PARAMETER MENU"]["METEOROLOGICAL PARAMETERS SUBMENU"]["Verti. diff. [m^2/s] (double)"] = DV
    # data["ADVANCED OPTIONS MENU"]["INITIAL CONTRAIL SIZE SUBMENU"]["Contrail Width Scaling Factor [-] (double)"] = 2.0
    data["ADVANCED OPTIONS MENU"]["GRID SUBMENU"]["NX (positive int)"] = NX
    data["ADVANCED OPTIONS MENU"]["GRID SUBMENU"]["NY (positive int)"] = NY
    data["SIMULATION MENU"]["OpenMP Num Threads (positive int)"] = NUM_THREADS
    
    if TEMP_PERTURB:
        data["METEOROLOGY MENU"]["TEMPERATURE PERTURBATION SUBMENU"]["Enable Temp. Pert. (T/F)"] = "T"
        data["METEOROLOGY MENU"]["TEMPERATURE PERTURBATION SUBMENU"]["Temp. Perturb. Amplitude (double)"] = TEMP_AMP
        data["METEOROLOGY MENU"]["TEMPERATURE PERTURBATION SUBMENU"]["Temp. Perturb. Timescale (min)"] = TEMP_TIMESTEP
        data["SIMULATION MENU"]["RANDOM NUMBER GENERATION SUBMENU"]["Force seed value (T/F)"] = "T"
        data["SIMULATION MENU"]["RANDOM NUMBER GENERATION SUBMENU"]["Seed value (positive int)"] = SEED_VALUE
    # Write the modified YAML back to the file
    with open(YAML_destination_path, "w") as file:
        yaml.dump(data, file, default_flow_style=False, indent=4)

    ## For overlay input
    overlay_source_path = f"{base_file_dir}/overlay-input.yaml"
    overlay_destination_dir = base_destination_dir #"f"/home/chinahg/GCresearch/contrailuncertainty/APCEMM_vs_CoCiP_vs_LES/APCEMM/epm_bypass/{test_id}"
    overlay_destination_path = os.path.join(overlay_destination_dir, "overlay-input.yaml")

    # Create the overlay destination directory if it doesn't exist
    if not os.path.exists(overlay_destination_dir):
        print(f"Creating directory: {overlay_destination_dir}")
        os.makedirs(overlay_destination_dir, exist_ok=False)

    # Copy the YAML file to the new directory
    shutil.copy(overlay_source_path, overlay_destination_path)

    # Read and modify the copied YAML file
    with open(overlay_destination_path, "r") as file:
        data = yaml.safe_load(file)  # Load YAML into a Python dictionary

    # Modify the YAML content
    data["SIMULATION MENU"]["External EPM NetCDF file"] = f"{base_file_dir}/{test_id}/epm-input-{test_identifier}.nc"

    # Write the modified YAML back to the file
    with open(overlay_destination_path, "w") as file:
        yaml.dump(data, file, default_flow_style=False, indent=4)

    # # Save the YAML and Overlay paths for bash inputs
    # YAML_paths.append(YAML_destination_path)
    # overlay_paths.append(overlay_destination_path)

    return YAML_destination_path, overlay_destination_path

# Run APCEMM with new epm-bypass.nc and YAML

In [15]:
# Run batches of APCEMM on slurm
bash_path = "/home/chinahg/GCresearch/contrailuncertainty/APCEMM_vs_CoCiP_vs_LES/APCEMM/run_apcemm.sh"

for i in range(num_sims):

    create_epm_input_netcdf(test_ids[i])
    YAML_path, overlay_path = create_yaml(i)

    arg1 = YAML_path + " " + overlay_path
    print(f"Submitting job for test ID: {test_ids[i]}")
    print(f"Input files are: {arg1}")

    export_args = f"ARG1={arg1}"

    # Update where the slurm output file is saved to
    with open(bash_path, "r") as file:
        bash_lines = file.readlines()

    # Modify the output file path in the bash script
    for j, line in enumerate(bash_lines):
        if "#SBATCH --job-name=" in line:
            bash_lines[j] = f"#SBATCH --job-name={test_identifier}_A{test_ids[i]}\n"
        if "#SBATCH -o" in line:
            bash_lines[j] = f"#SBATCH -o /home/chinahg/GCresearch/contrailuncertainty/APCEMM_vs_CoCiP_vs_LES/APCEMM/testing/{test_identifier}/{test_ids[i]}/slurm-%j-out\n"
            break
        if "#SBATCH --cpus-per-task=" in line:
            bash_lines[j] = f"#SBATCH --cpus-per-task={NUM_THREADS}\n"
            break

    # Write the modified bash script back to the file
    with open(bash_path, "w") as file:
        file.writelines(bash_lines)

    # Submit the job and get the job ID
    lib.submit_job_and_get_id(bash_path, "has_args", export_args)

    time.sleep(5)  # Optional: wait a bit before submitting the next job

Running binning test...
Original number of bins: 38, New number of bins: 1000, Original K: 1.21644, New K: 1.00728
New bin edges: [5.00000000e-08 5.03641503e-08 5.07309527e-08 ... 6.98512402e-05
 7.03599672e-05 8.55897456e-05]
New bin centers: [5.54110100e-08 5.58145687e-08 5.62210666e-08 5.66305250e-08
 5.70429654e-08 5.74584097e-08 5.78768797e-08 5.82983974e-08
 5.87229850e-08 5.91506648e-08 5.95814595e-08 6.00153916e-08
 6.04524841e-08 6.08927600e-08 6.13362423e-08 6.17829546e-08
 6.22329202e-08 6.26861630e-08 6.31427067e-08 6.36025754e-08
 6.40657934e-08 6.45323850e-08 6.50023747e-08 6.54757874e-08
 6.59526480e-08 6.64329816e-08 6.69168134e-08 6.74041690e-08
 6.78950740e-08 6.83895542e-08 6.88876358e-08 6.93893449e-08
 6.98947079e-08 7.04037515e-08 7.09165025e-08 7.14329878e-08
 7.19532347e-08 7.24772706e-08 7.30051230e-08 7.35368198e-08
 7.40723889e-08 7.46118586e-08 7.51552572e-08 7.57026135e-08
 7.62539561e-08 7.68093141e-08 7.73687168e-08 7.79321937e-08
 7.84997744e-08 7.907148